# Shortcut Models

推荐你读 https://arxiv.org/abs/2410.12557 One Step Diffusion via Shortcut Models 这是 Shortcut 模型原文。

我们提出一个简单纯良的想法，为模型输入时间推理步长 $d$，这样模型可以本质地知晓当前推理的幅度。

## 基本逻辑

Shortcut 模型的核心任务仍是拟合时间平均矢量场，这与 Meanflow 极度相似。与 Meanflow 简单地不同的是，Meanflow 选择为模型输入起始时间步与终点时间步，Shortcut 的做法则是为模型输入起始时间步与时间推理距离。他们都要求模型拟合这一段时间内的平均矢量场。

我们保持来自 Flow Matching 那一章的记号。简而言之就是
$$x'_{t+d} = x_t + s(x_t,t,d)d$$
当 $d \to 0$，这就退化为原始 FM。

更多的，我们拥有天生的一致性关系
$$s(x_t,t,2d) = s(x_t,t,d)/2 + s(x'_{t+d},t+d,d)/2$$

下面这张图展示了 Shortcut 模型的基本逻辑。平均矢量场的本质是捷径。

<img src="./assets/Shortcut.png" width="1000" height="350">

所以训练其实非常简单，对于时间步长 $d \sim p(d)$ 采样即可。我们有一定概率直接采样 $d=0$，另外依据一个概率密度函数采样 $d > 0$。

Shortcut 模型与 Meanflow 模型最大的实际上就在训练方式。Meanflow 的核心思想是直接使用一阶微分与瞬时速度拟合时间平均矢量场 $u^\theta_{t \to r}(x_t)$。但是 Shortcut 则选择直接在 $d=0$ 拟合瞬时矢量场，在 $d >0$ 时使用一致性关系拟合更大的推理步长下的平均矢量场。这实际上极大程度依赖了 $s_\theta(x_t,t,\cdot)$ 关于步长 $d$ 的连续性。

我们自然地给出目标函数
$$\mathcal{L}^{\text{S}}(\theta) = \mathbb{E}_{x_0 \sim \mathcal{N}, x_1 \sim D, (t, d) \sim p(t, d)} \left[ \underbrace{\|s_\theta(x_t, t, 0) - (x_1 - x_0)\|^2}_{\text{Flow-Matching}} + \underbrace{\|s_\theta(x_t, t, 2d) - s_{\text{target}}\|^2}_{\text{Self-Consistency}} \right]$$
此处 $s_{\text{target}} = s_\theta(x_t, t, d)/2 + s_\theta(x'_{t+d}, t, d)/2 $ 并且 $ x'_{t+d} = x_t + s_\theta(x_t, t, d)d$。

左边是原始 FM 拟合瞬时速度的绝对真实项，右边是一致性损失项。

## 训练

在 $d=0$ 的训练下，我们实际上就是在训练原始 FM 模型。在 $d>0$ 的情况下，我们希望模型对于大步平均矢量场的学习保持对小步平均矢量场的一致性。所以我们应该在训练中逐步增大步长，我们为此选择一种以 $2$ 为倍数的步长增加方式。

具体来说，在每个 Batch 当中，我们将前 $1-k$ 部分设置为 $d=0$，后 $k$ 部分则将 $d$ 从 $(1/128. 1/64, ..., 1/2,1)$ 中抽样。实际中，$k = 1/4$。

所以完整的训练是，首先采样噪声 $x_0 \sim \mathcal{N}(0,\mathbf{I})$，真实数据点 $x_1 \sim D$，时间步 $t \sim \mathcal{U}[0,1]$，如上的步长 $d$。

前向加噪得到 $x_t = (1-t)x_0 + tx_1$。

对于原始逼近真实瞬时矢量场部分，我们直接将回归目标设置为 $x_1 -x_0$。输入模型得到 $s_\theta(x_t,t,0)$，做 FM 的训练。

对于一致性目标，我们首先运行三次网络得到 $s_\theta(x_t,t,d)$ 与 $s_\theta(x_{t+d},t+d,d)$ 与 $s_\theta(x_{t},t,2d)$，然后按照 MSE 距离计算损失函数。

最终一个 Batch 的损失是所有损失加权取平均，反向传播更新梯度。

一个工程上的优化是权重衰减。简而言之就是训练初始时模型的一致性预测可能完全无意义，于是其权重理应极小。随着训练的进行，一致性损失应当被逐渐放大。

更多的，由于一致性损失依赖模型自身的推理结果，这很有可能造成训练目标的偏移。因此一个技巧是在构建一致性目标时使用 EMA 权重保证训练稳定性。

下面是完整训练算法。

$$\begin{array}{ll}
\hline
\textbf{Algorithm 1 } \text{Shortcut Model Training} \\
\hline
\textbf{while } \text{not converged} \textbf{ do} \\
\quad x_0 \sim \mathcal{N}(0, I), \, x_1 \sim D, \, (d, t) \sim p(d, t) \\
\quad x_t \leftarrow (1 - t) x_0 + t x_1 & \text{Noise data point} \\
\quad \textbf{for } \text{first } k \text{ batch elements} \textbf{ do} \\
\qquad s_{\text{target}} \leftarrow x_1 - x_0 & \text{Flow-matching target} \\
\qquad d \leftarrow 0 \\
\quad \textbf{for } \text{other batch elements} \textbf{ do} \\
\qquad s_t \leftarrow s_{\theta^-}(x_t, t, d) & \text{First small step} \\
\qquad x_{t+d} \leftarrow x_t + s_t d & \text{Follow ODE} \\
\qquad s_{t+d} \leftarrow s_{\theta^-}(x_{t+d}, t + d, d) & \text{Second small step} \\
\qquad s_{\text{target}} \leftarrow \text{stopgrad}(s_t + s_{t+d}) / 2 & \text{Self-consistency target} \\
\quad \theta \leftarrow \nabla_{\theta} \| s_{\theta}(x_t, t, 2d) - s_{\text{target}} \|^2 \\
\hline
\end{array}$$

## 推理

推理非常简单。我们选定推理步数之后，将时间区间 $[0,1]$ 分割为对应的步数，然后逐步应用 Shortcut 模型的推理公式即可。

值得一提，作者指出在 $d>0$ 时不建议开启 CFG 推理，原因是可能来带更大的步长偏差。

下面是完整推理算法。

$$\begin{array}{l}
\hline
\textbf{Algorithm 2 } \text{Sampling} \\
\hline
x \sim \mathcal{N}(0, I) \\
d \leftarrow 1 / M \\
t \leftarrow 0 \\
\textbf{for } n \in [0, \dots, M - 1] \textbf{ do} \\
\quad x \leftarrow x + s_{\theta}(x, t, d) d \\
\quad t \leftarrow t + d \\
\textbf{return } x \\
\hline
\end{array}$$

## Shortcut Models 总结

Shortcut Models 核心思想极其简单，就是加入一个时间步长参数 $d$，并且依靠一致性进行训练。我们已经说完了。

一个有趣的事实是，已经训练好的 Shortcut 模型使用 FM 的推理方式在相同步数下优于原始 FM。这可能是因为一致性损失起到某种正则化效果。

还有一件事，就是 Shortcut 模型对于插值展现出一种平滑的变化效果，这表明模型学习到了噪声空间与真实像素空间之间的映射关系。更具体的，作者抽取噪声点 $(x_0^0, x_0^1)$ 进行方差保持的插值 
$$x_0^n = n x_0^0 + \sqrt{1 - n^2} x_0^1 \quad (n \in [0, 1])$$
根据新的位置进行推理，对比原本噪声点推理结果，最终发现结果符合插值的预期。

# Shortcut Models 的改进与抽象

我们来看另一篇高度相关的成果 ExplicitShortCut。ESC 提出了少步模型一个统一框架，并且提出一些新的工程技巧。

关于 ESC 与 UCGM 之间的关系，简单来说就是 ESC 聚焦于统一各类少步模型框架，如 CM 与 Shortcut Models，而 UCGM 则聚焦于统一多部模型与少步模型。

推荐你读 https://arxiv.org/abs/2512.11831 On the Design of One-step Diffusion via Shortcutting Flow Paths 这是 ESC 原文。

## 基本逻辑

我们简称 Shortcut Models 为 SC 或 SCD。

我们称前向加噪时预设的方式为流路径，将流路径上两点之间映射称为流映射。换言之，流路径遵守方程
$$\dot{x}_t = v_t(x_t)$$
而一个流映射 $X_{t,r}( \cdot )$ 将 $x_t$ 送到 $x_r$。

首先我们认为，当下的少步生成模型本质都在学习一个时间平均矢量场 $u_{t,r}(x_t)$。更具体的
$$X_{t,r}(x_t) = x_t + (r-t) \cdot u_{t,r}(x_t)$$
并且
$$ u_{t,r}(x_t) = \frac{1}{r-t} \int_t^r v_\tau(x_\tau) d\tau $$


想要达到少步生成的目的，我们拥有两种推理方法。对于瞬时矢量场模型，如 Flow Matching 或 DDIM 模型，我们使用瞬时速度来近似一段时间内的平均速度，即
$$X_{t,r}(x_t) \approx \text{DDIM}(x_t, v_t, t, r) = \bar{\alpha}_{t,r} x_t + \bar{\beta}_{t,r} v_t$$
而对于平均时间矢量场模型，如 Meanflow 模型，我们天然地利用定义进行推理
$$x_r = x_t + (r-t) \cdot u_{t,r}(x_t)$$

而学习这个平均时间矢量场的方式，也有两种方法。首先是 Meanflow 的做法，直接学习一个解析的微分式
$${u(z_t, r, t)} = {v(z_t, t)} - (t - r) {\frac{d}{dt} u(z_t, r, t)}$$
损失函数则是
$$\mathcal{L}^{\text{MF}}_\theta = \mathbb{E} \| V_\theta  - v(z_t, t) \|_2^2 $$
其中 $ V_\theta = {u_\theta (z_t, r, t)} + (t - r) {\frac{d}{dt} u_\theta (z_t, r, t)}$。

其次是 Shortcut 模型的做法，模型首先学习瞬时矢量场，其次在此基础上使用一致性目标学习平均矢量场
$$\mathcal{L}^{\text{SC}}(\theta) = \mathbb{E}_{x_0 \sim \mathcal{N}, x_1 \sim D, (t, d) \sim p(t, d)} \left[ \underbrace{\|s_\theta(x_t, t, 0) - (x_1 - x_0)\|^2}_{\text{Flow-Matching}} + \underbrace{\|s_\theta(x_t, t, 2d) - s_{\text{target}}\|^2}_{\text{Self-Consistency}} \right]$$
此处 $s_{\text{target}} = s_\theta(x_t, t, d)/2 + s_\theta(x'_{t+d}, t, d)/2 $ 并且 $ x'_{t+d} = x_t + s_\theta(x_t, t, d)d$。


我们现在指出统一的损失函数。首先我们利用一个关键的恒等式
$$X_{s,r}(X_{t,s}(x_t)) = X_{t,r}(x_t)$$
损失函数是
$$\arg \min_\theta \mathbb{E} \Big[ w(r, s, t) \cdot D\big( \underbrace{X_{t,r}^\theta(x_t)}_{\text{one-step prediction}} , \ \underbrace{\text{sg}(\hat{X}_{s,r} \circ \hat{X}_{t,s}(x_t))}_{\text{two-step target}} \big) \Big]  \quad (*)$$
其中 $D( \cdot )$ 是一个度量，一般是 L2 度量或 $LPIPS$ 度量。

## 对应

我们来说说这个对应。首先我们将 Meanflow 之类直接学习微分形式的模型称为 Continuous-time shortcut models，将 Shortcut 之类学习一个离散形式的模型称为 Discrete-time shortcut models。

我们先说 CTSC。

当我们设置 $s = t - dt$，$\hat{X}_{t-dt, r} \big( \hat{X}_{t, t-dt}(x_t) \big)$ 即为
$$\text{Target} = \hat{x}_{t - dt} + \big(r - (t - dt)\big) \cdot u_\theta(\hat{x}_{t-dt}, r, t - dt)$$
其中
$$\hat{x}_{t - dt} = x_t - dt \cdot v(x_t)$$
或者我们完全展开
$$\text{Target} = (x_t - dt \cdot v(x_t)) + (r - t + dt) \cdot u_\theta(x_t - dt \cdot v(x_t), r, t - dt)$$
关于最后一项
$$u_\theta(x_t - dt \cdot v(x_t), r, t - dt) \approx (r - t)u_\theta(x_t, r, t) + dt \cdot u_\theta(x_t, r, t) - (r - t)dt \cdot \frac{du_\theta}{dt}$$

最后全部合并，得到
$$\text{Two-step Target} = \underbrace{\big[ x_t + (r - t)u_\theta(x_t, r, t) \big]}_{\text{one-step prediction } X_{t,r}^\theta} + dt \cdot \left[ u_\theta(x_t, r, t) - v(x_t) - (r - t)\frac{du_\theta}{dt} \right]$$
这意味着当我们将单步目标与多步目标都代入，损失函数形式就是
$$\mathcal{L}^{MF} = D\Big ( u_\theta(x_t, r, t) - \left( v(x_t) + (r - t)\frac{du_\theta}{dt} \right) \Big)$$
这就是 Meanflow 的损失函数。

现在我们来说 DTSC。

我们设定被映射的时间点分别是 $t, t+d, t+2d$。其实损失函数的对应非常简单，我们直接代入
$$\mathcal{L}^{SC} = D\Big( \Big( x_t + 2d \cdot s_\theta(x_t, t, 2d) \Big) - \text{sg}\Big( x_t + d \cdot \left[ s_\theta(x_t, t, d) + s_\theta(x'_{t+d}, \ t+d, \ d) \right] \Big) \Big)$$
化简一下
$$\mathcal{L}^{SC} = 2d \cdot  D\Big( s_\theta(x_t, t, 2d) - \frac{1}{2}\text{sg}\Big( s_\theta(x_t, t, d) + s_\theta(x'_{t+d}, \ t+d, \ d) \Big) \Big )$$
这就是 Shortcut 模型损失函数的一致性项。

我还想提醒一点，如果在统一损失函数 $(*)$ 中取 $r \to t$，这个损失函数甚至可以化为 Flow Matching 的形式。这是因为
$$\mathcal{L}^{FM} = D \Big( \Big( x_t + dt \cdot v_\theta(x_t, t) \Big) - \Big( x_t + dt \cdot (x_1 - x_0) \Big) \Big)$$
这就是
$$\mathcal{L}^{FM} =  D \Big( v_\theta(x_t, t) - (x_1 - x_0) \Big) $$

所以 Shortcut 模型非常有趣，实际上他的前半部分瞬时速度损失来自 FM 的形式，后半部分一致性损失则是 DTSC 的形式。总的损失可以统一为损失 $(*)$ 的加权平均形式。但是我们更关注其一致性损失形式，所以还是将其归入 DTSC。

现在我们为所有少步生成模型建立了统一的框架。他们都学习一种时间尺度上的平均矢量场，并且共享同一套损失函数，拥有高度类似的推理方式。

我们建立统一框架的目的不仅仅是抽象，还有对于各类因素的解耦。我们希望探究清楚哪部分措施提升了最终的性能。

## 思考与问题

正式开始做出一些优化之前，我们先来思考现有模型的缺陷与问题。

我们指出一些事实。第一，DTSC 模型训练至良好的难度高于 CTSC 模型。原因是，我们可以推导 DTSC 模型的预测概率分布与真实概率分布之间 Wasserstein-2 距离可以控制在模型损失加上时间步差距量级，也就是说
$$W_2^2(p_0, p_0^{\theta}) \le C_1 \mathcal{L}_{\text{dtsc}}(\theta) + C_2(t - s)$$
然而，CTSC 模型概率分布与真实概率分布之间 Wasserstein-2 距离则可以进一步控制在纯粹模型损失量级
$$W_2^2(p_0, p_0^{\theta}) \le C_3 \mathcal{L}_{\text{ctsc}}(\theta)$$
这意味着两者均训练良好情况下，CTSC 模型理论性能高于 DTSC 模型。

第一个事实比较容易理解。这是因为 DTSC 的损失函数没有对最短宏观步长之下的损失要求优化，意味着模型对于 $d < d_{min}$ 时精度控制的学习是不足的。而 CTSC 模型，如 Meanflow，采取直接微分或者近似微分方式计算损失，这保证了最终训练效果的控制。

第二个事实是，模型的学习目标永远是偏移的。这很直观，因为无论是 Meanflow 还是 SC 模型，其损失函数中预测目标永远是将自身预测项作为回归目标。而自身预测项是高方差的，这造成了学习的困难。

下面这张图展示了这个事实。理想情况下，DTSC 的两步预测永远符合一步预测，CTSC 学习到的平均矢量场永远真实。实际上完全不是这样。

<img src="./assets/D_C.png" width="1000" height="400">

第三件事实是，蒸馏现有 FM 模型永远比从零开始训练更加容易。这是因为蒸馏一个已经存在的 FM 模型相当于已经掌握了真实的瞬时速度场，我们以此为基础学习平均矢量场是几乎无偏的。

现在我们尝试解决这些问题。

## 优化

首先决定几个关键的选择。

第一，直线条件概率路径还是余弦或者其他概率路径？毫无疑问直线路径，首先其形式更简单，其次其诱导的边缘矢量场曲率更低。

第二，DTSC 还是 CTSC？答案是 CTSC，我们刚刚已经说出其在理想情况下的收敛优点。实际上实际测试中，DTSC 性能确实劣于 CTSC。

第三个抉择详细说说，就是关于学的平均矢量场的终点时间的选择。一个选择是模型直接学习任意点到终点时间步 $t=0$ 映射 $X_{t \to 0}$。另一个选择则是模型也需要学习任意点到任意点映射 $X_{t \to r}$，其中 $0 \le t \le r \le 1$。前者是 CM 和传统 Diffusion 模型的做法，后者是 Shortcut 和 MF 模型的做法。训练前期，直接学习到终点的映射收敛非常快，但是达到性能上限还是需要学习到任意时间点映射。

最终作者选择是训练初期较多固定 $r=0$，之后逐渐过渡到 MeanFlow 随机 $r$ 采样

### Plug-in Velocity

现在我们正式提出一些改进。首先第一个改进被称为 Plug-in Velocity。简单来说，我们知道原始 Flow Matching 关于边缘矢量场损失等价条件矢量场损失。更多的，这个等价关系实际上是指出了训练计算损失的最小单位是单独的一条条件矢量场。这个条件矢量场最大的问题就是方差太大，因为我们尝试统计许许多多条件矢量场时，他们无可避免地互相干扰。

现在我们将这个最小单位加大，加大为一个 Batch 中所有样本构造一个经验边缘矢量场。这个边缘矢量场实际上就是考虑且仅仅考虑一个 Batch 内所有条件矢量场同时存在得到的边缘矢量场。

更具体的，对于一个大小为 $N$ 的 Batch 样本集 $\{y^{(i)}\}_{i=1}^N$，我们在局部构造一个边缘速度场 $v_t^*$
$$v_t^*(x_t|\{y^{(i)}\}_{i=1}^N) = \sum_i \underbrace{\frac{\mathcal{N}(x_t; \alpha_t y^{(i)}, \sigma_t^2 \mathbf{I})}{\sum_j \mathcal{N}(x_t; \alpha_t y^{(j)}, \sigma_t^2 \mathbf{I})}}_{\text{高斯似然权重 } w_i} \underbrace{\left( \dot{\alpha}_t y^{(i)} + \frac{\dot{\sigma}_t}{\sigma_t}(x_t - \alpha_t y^{(i)}) \right)}_{\text{单一样本的条件速度 } v_{t|y^{(i)}}}$$

直观地说，对于一个给定的 $x_t$，batch 里每个训练样本 $y_i$ 都可能是诱导它的原始数据。于是可以反推出
$$\epsilon_i = \frac{x_t - (1 - t)y_i}{t}$$
然后根据 $\epsilon_i$ 在标准高斯下的概率给出权重
$$w_i \propto \mathcal{N}(\epsilon_i; 0, I)$$
最后加权平均得到
$$v_{\text{plugin}}(x_t) = \sum_i w_i (\epsilon_i - y_i)$$
实际上我们对 Batch 内所有 $x_t$ 和所有 $x$ 两两配对，算出可能的 $\epsilon$ 与权重，再加权条件速度。

好处是巨大的。将 $v_{t|0}$ 替换为 $v_t^*$ 后，监督信号的方差降低了 $\mathcal{O}(1 - 1/N)$。当 Batch Size $N$ 足够大时，随机抽样带来的单点抖动被加权平均几乎彻底抹平，回归靶子变得极其稳定。

但是代价不是没有。这样会引入 $\mathcal{O}(1/N)$ 的偏置。因为局部的一个 Batch 无法完美等价于全局真实的完整数据分布，用 Batch 的经验边缘速度去近似真实的全局边缘速度，必然存在系统误差。

我们详细解释一下为什么会引入偏置。原始 FM 学习的条件矢量场是无偏的，并且诱导的理想的边缘矢量场是
$$v_t(x_t) = \mathbb{E}[\epsilon - x_0 \mid x_t]$$
但是我们现在直接认为，一个 Batch 之内所有数据就可以估计出一个点的边缘矢量场，现实是
$$v_t^{Batch} \neq v_t$$
并且我们不得不这么做，因为估计边缘矢量场注定需要做一次归一化除法，而做完除法之后的期望就无法做到无偏。

所以我们说这样计算的是 Batch 这个局部中的边缘矢量场，不是完整数据分布里的边缘矢量场

### Plug-in Velocity under CFG

一个问题是，一个 Batch 内数据也许无法做到全部带有同一标签，这意味着直接求出局部边缘矢量场是不可行的。

所以改进很简单，一个 Batch 内数据全部被要求来自同一标签即可。更多的，我们引入一个 Plug-in 概率 $p_{plug-in}$，有时我们选取局部边缘矢量场，有时我们直接使用原始条件矢量场。

### 时间采样

这关于上述的第三个选择。最终作者的做法是开始时以概率 $p_{fix0}$ 设置 $r=0$，其余情况遵从 Meanflow 的时间步采样方法。更多的，$p_{fix0}$ 本身遵从一个随着训练轮次进行的余弦渐变调度。

### 自适应损失权重

我们在 iMF 中提到的自适应损失权重被在这里使用。这是一个广泛有效的技巧。

最终，我们选取 Baseline 是 MeanFlow 算法在 SiT-B/2 网络下训练，FID 是 6.09。配合我们上述提到诸多技巧之后，FID 降低到 5.77。更多的，每加上一种技巧，FID 都会降低。这意味着每个技巧都是有效的。 

顺便解释一下什么是 SiT，实际上是 Scalable Interpolant Transformers。我需要提醒一件事，我们之前其实可能提到 SiT 很多次了。这是因为一部人觉得 DiT 架构放在 Flow Matching 算法语境下不能叫 Diffusion Transformer，因为原始 DiT 是为 DDIM 设计的而不是 FM 直线路径，并且一个观点是 FM 不算传统 Diffusion。所以有研究者验证 DiT 在 FM 框架下有效性之后，一个别称 SiT 被提出。总之他们没什么核心区别。

# 总结

本章我们讲述了少步模型生成的大一统，并且指出过往方法的缺点。

实际上我们已经基本介绍完了关于少步生成领域主要内容。下面我想说一些建立直觉的成果，他们给出的成果是非常助于理解的。

我为你介绍 Spherical Flow Matching 与其衍生的 Riemannian Meanflow。我们指出，图像生成的本质是建立高斯球壳到真实数据流形之间路径。